<a href="https://colab.research.google.com/github/trang1981/ELAPS/blob/main/Notebooks/Santander_PLACEBO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

XỨ LÝ PLACEBO VÀ GỘP DỮ LIỆU CHO TẬP DL Santander

In [ ]:
# ==========================================================
# SANTANDER PLACEBO-CUTOFF DATA CONSTRUCTION
# Adopters: retain observed acquisition cutoff
# Non-adopters: receive pseudo-cutoff sampled from adopter-tenure distribution
# ==========================================================

# ==========================================================
# 1. Mount Google Drive
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

# ==========================================================
# 2. Imports
# ==========================================================

import os
import gc
import numpy as np
import pandas as pd

from scipy.stats import ks_2samp

# ==========================================================
# 3. Configuration
# ==========================================================

INPUT_FILENAME = "Santander_CreditCard.csv"

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "Santander_Placebo_Cutoff_Datasets"
)

SEEDS = [42, 52, 62, 72, 82]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 19 variables taking the last available pre-cutoff value
LAST_VALUE_FEATURES = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "age",
    "ind_nuevo",
    "antiguedad",
    "indrel",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "conyuemp",
    "indfall",
    "tipodom",
    "cod_prov",
    "nomprov",
    "ind_actividad_cliente",
    "renta",
    "segmento",
]

# One variable taking the first available pre-cutoff value
FIRST_VALUE_FEATURES = [
    "canal_entrada",
]

MODEL_FEATURES = LAST_VALUE_FEATURES + FIRST_VALUE_FEATURES

print("Số đặc trưng mô hình:", len(MODEL_FEATURES))

if len(MODEL_FEATURES) != 20:
    raise ValueError(
        f"Danh sách đặc trưng phải có đúng 20 biến, "
        f"nhưng hiện có {len(MODEL_FEATURES)} biến."
    )

# ==========================================================
# 4. Find input file automatically
# ==========================================================

input_file = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if INPUT_FILENAME in files:
        input_file = os.path.join(root, INPUT_FILENAME)
        break

if input_file is None:
    raise FileNotFoundError(
        f"Không tìm thấy {INPUT_FILENAME} trong Google Drive."
    )

print("\nĐang đọc file:")
print(input_file)

# ==========================================================
# 5. Load raw data
# ==========================================================

df = pd.read_csv(input_file, low_memory=False)

print("\nKích thước dữ liệu gốc:", df.shape)

required_columns = (
    [
        "ncodpers",
        "fecha_dato",
        "fecha_alta",
        "ind_tjcr_fin_ult1",
    ]
    + MODEL_FEATURES
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        "Dữ liệu thiếu các cột bắt buộc:\n"
        + "\n".join(missing_columns)
    )

# ==========================================================
# 6. Standardize date and target variables
# ==========================================================

df["fecha_dato"] = pd.to_datetime(
    df["fecha_dato"],
    errors="coerce"
)

df["fecha_alta"] = pd.to_datetime(
    df["fecha_alta"],
    errors="coerce"
)

df["ind_tjcr_fin_ult1"] = pd.to_numeric(
    df["ind_tjcr_fin_ult1"],
    errors="coerce"
).fillna(0).astype("int8")

df = df.dropna(
    subset=[
        "ncodpers",
        "fecha_dato",
        "fecha_alta",
    ]
).copy()

# ==========================================================
# 7. Keep only the first relationship year
# ==========================================================

df["end_observation"] = (
    df["fecha_alta"] + pd.DateOffset(years=1)
)

df = df[
    (df["fecha_dato"] >= df["fecha_alta"])
    & (df["fecha_dato"] < df["end_observation"])
].copy()

df = df.sort_values(
    ["ncodpers", "fecha_dato"]
).reset_index(drop=True)

print("Sau khi giữ năm quan hệ đầu tiên:", df.shape)
print("Số khách hàng:", df["ncodpers"].nunique())

# ==========================================================
# 8. Construct customer-level metadata
# ==========================================================

# First credit-card acquisition date within the first year
event_dates = (
    df.loc[
        df["ind_tjcr_fin_ult1"].eq(1),
        ["ncodpers", "fecha_dato"]
    ]
    .groupby("ncodpers")["fecha_dato"]
    .min()
    .rename("event_date")
)

customer_meta = (
    df.groupby("ncodpers", as_index=False)
    .agg(
        fecha_alta=("fecha_alta", "first"),
        first_snapshot=("fecha_dato", "min"),
        last_snapshot=("fecha_dato", "max"),
    )
)

customer_meta = customer_meta.merge(
    event_dates,
    on="ncodpers",
    how="left",
)

customer_meta["TARGET"] = (
    customer_meta["event_date"]
    .notna()
    .astype("int8")
)

# ==========================================================
# 9. Remove invalid adopter cases
# ==========================================================

# An adopter must have at least one snapshot strictly before
# the observed acquisition event.
invalid_adopter = (
    customer_meta["TARGET"].eq(1)
    & (
        customer_meta["first_snapshot"]
        >= customer_meta["event_date"]
    )
)

print(
    "\nAdopters không có dữ liệu trước event:",
    int(invalid_adopter.sum())
)

valid_customer_ids = customer_meta.loc[
    ~invalid_adopter,
    "ncodpers"
]

customer_meta = customer_meta.loc[
    ~invalid_adopter
].copy()

df = df[
    df["ncodpers"].isin(valid_customer_ids)
].copy()

# ==========================================================
# 10. Compute adopter-tenure distribution
# ==========================================================

adopter_mask = customer_meta["TARGET"].eq(1)

customer_meta.loc[
    adopter_mask,
    "observed_tenure_days"
] = (
    customer_meta.loc[adopter_mask, "event_date"]
    - customer_meta.loc[adopter_mask, "fecha_alta"]
).dt.days

adopter_tenures = (
    customer_meta.loc[
        adopter_mask,
        "observed_tenure_days"
    ]
    .dropna()
    .astype(int)
)

# Only strictly positive tenures are admissible
adopter_tenures = adopter_tenures[
    adopter_tenures > 0
].to_numpy()

adopter_tenures.sort()

if len(adopter_tenures) == 0:
    raise ValueError(
        "Không có adopter tenure hợp lệ để tạo pseudo-cutoff."
    )

print("\nPhân phối adopter tenure:")
print(pd.Series(adopter_tenures).describe())

# ==========================================================
# 11. Feasible tenure range for each non-adopter
# ==========================================================

non_adopter_mask = customer_meta["TARGET"].eq(0)

# Minimum tenure required to retain at least the first snapshot.
# Because records are selected with fecha_dato < cutoff,
# cutoff must be later than first_snapshot.
customer_meta.loc[
    non_adopter_mask,
    "min_feasible_tenure"
] = (
    customer_meta.loc[
        non_adopter_mask,
        "first_snapshot"
    ]
    - customer_meta.loc[
        non_adopter_mask,
        "fecha_alta"
    ]
).dt.days + 1

# Maximum tenure allowed by available observation history.
# +1 allows the final available snapshot to be retained because
# feature records must satisfy fecha_dato < pseudo_cutoff.
customer_meta.loc[
    non_adopter_mask,
    "max_feasible_tenure"
] = (
    customer_meta.loc[
        non_adopter_mask,
        "last_snapshot"
    ]
    - customer_meta.loc[
        non_adopter_mask,
        "fecha_alta"
    ]
).dt.days + 1

customer_meta[
    "min_feasible_tenure"
] = customer_meta[
    "min_feasible_tenure"
].fillna(1).astype(int)

customer_meta[
    "max_feasible_tenure"
] = customer_meta[
    "max_feasible_tenure"
].fillna(1).astype(int)

# ==========================================================
# 12. Sample feasible pseudo-tenures efficiently
# ==========================================================

def sample_feasible_placebo_tenures(
    min_tenures,
    max_tenures,
    empirical_tenures,
    seed,
):
    """
    Sample one placebo tenure for each non-adopter from the
    empirical adopter-tenure distribution.

    Sampling is conditional on:
        min_feasible_tenure <= sampled tenure
        <= max_feasible_tenure

    Returns
    -------
    sampled_tenures : np.ndarray
    fallback_flags  : np.ndarray
        1 where no empirical adopter tenure was available within
        the customer's feasible interval and a nearest feasible
        tenure had to be used.
    """

    rng = np.random.default_rng(seed)

    min_tenures = np.asarray(min_tenures, dtype=int)
    max_tenures = np.asarray(max_tenures, dtype=int)

    sampled = np.empty(
        len(min_tenures),
        dtype=int,
    )

    fallback = np.zeros(
        len(min_tenures),
        dtype="int8",
    )

    for i, (lower, upper) in enumerate(
        zip(min_tenures, max_tenures)
    ):
        if upper < lower:
            upper = lower

        # Empirical adopter tenures satisfying feasibility
        left = np.searchsorted(
            empirical_tenures,
            lower,
            side="left",
        )

        right = np.searchsorted(
            empirical_tenures,
            upper,
            side="right",
        )

        if right > left:
            random_index = rng.integers(
                low=left,
                high=right,
            )

            sampled[i] = empirical_tenures[
                random_index
            ]

        else:
            # Rare fallback:
            # choose the closest adopter tenure and clip it to the
            # customer's feasible history.
            insertion_point = np.searchsorted(
                empirical_tenures,
                lower,
                side="left",
            )

            candidate_indices = []

            if insertion_point > 0:
                candidate_indices.append(
                    insertion_point - 1
                )

            if insertion_point < len(
                empirical_tenures
            ):
                candidate_indices.append(
                    insertion_point
                )

            if candidate_indices:
                candidates = empirical_tenures[
                    candidate_indices
                ]

                closest = candidates[
                    np.argmin(
                        np.abs(candidates - lower)
                    )
                ]

                sampled[i] = int(
                    np.clip(
                        closest,
                        lower,
                        upper,
                    )
                )
            else:
                sampled[i] = lower

            fallback[i] = 1

    return sampled, fallback


# ==========================================================
# 13. Reconstruct one placebo dataset
# ==========================================================

def construct_placebo_dataset(
    longitudinal_df,
    meta_df,
    seed,
):
    """
    Adopters:
        cutoff = observed first acquisition date

    Non-adopters:
        cutoff = fecha_alta + sampled placebo tenure

    All 20 features are reconstructed using only observations
    strictly preceding the assigned cutoff.
    """

    meta_seed = meta_df.copy()

    non_mask = meta_seed["TARGET"].eq(0)

    sampled_tenure, fallback_flag = (
        sample_feasible_placebo_tenures(
            min_tenures=meta_seed.loc[
                non_mask,
                "min_feasible_tenure",
            ].to_numpy(),
            max_tenures=meta_seed.loc[
                non_mask,
                "max_feasible_tenure",
            ].to_numpy(),
            empirical_tenures=adopter_tenures,
            seed=seed,
        )
    )

    meta_seed["assigned_tenure_days"] = np.nan
    meta_seed["fallback_sampling"] = 0

    # Adopters retain true tenure and true cutoff
    meta_seed.loc[
        meta_seed["TARGET"].eq(1),
        "assigned_tenure_days",
    ] = meta_seed.loc[
        meta_seed["TARGET"].eq(1),
        "observed_tenure_days",
    ]

    meta_seed.loc[
        meta_seed["TARGET"].eq(1),
        "assigned_cutoff",
    ] = meta_seed.loc[
        meta_seed["TARGET"].eq(1),
        "event_date",
    ]

    # Non-adopters receive pseudo-tenure and pseudo-cutoff
    meta_seed.loc[
        non_mask,
        "assigned_tenure_days",
    ] = sampled_tenure

    meta_seed.loc[
        non_mask,
        "fallback_sampling",
    ] = fallback_flag

    meta_seed.loc[
        non_mask,
        "assigned_cutoff",
    ] = (
        meta_seed.loc[
            non_mask,
            "fecha_alta",
        ]
        + pd.to_timedelta(
            sampled_tenure,
            unit="D",
        )
    )

    meta_seed[
        "assigned_tenure_days"
    ] = meta_seed[
        "assigned_tenure_days"
    ].astype(int)

    # Merge assigned cutoff into longitudinal records
    working = longitudinal_df.merge(
        meta_seed[
            [
                "ncodpers",
                "TARGET",
                "assigned_cutoff",
                "assigned_tenure_days",
                "fallback_sampling",
            ]
        ],
        on="ncodpers",
        how="inner",
        validate="many_to_one",
    )

    # Strictly pre-cutoff records only
    working = working[
        working["fecha_dato"]
        < working["assigned_cutoff"]
    ].copy()

    working = working.sort_values(
        ["ncodpers", "fecha_dato"]
    )

    # ------------------------------------------------------
    # Aggregate the 20 leakage-aware features
    # ------------------------------------------------------

    aggregation_rules = {
        col: "last"
        for col in LAST_VALUE_FEATURES
    }

    aggregation_rules.update({
        col: "first"
        for col in FIRST_VALUE_FEATURES
    })

    customer_level = (
        working
        .groupby(
            "ncodpers",
            as_index=False,
            sort=False,
        )
        .agg(aggregation_rules)
    )

    # Audit variables
    audit_values = (
        working
        .groupby(
            "ncodpers",
            as_index=False,
            sort=False,
        )
        .agg(
            fecha_alta=("fecha_alta", "first"),
            last_snapshot_date=(
                "fecha_dato",
                "max",
            ),
            assigned_cutoff=(
                "assigned_cutoff",
                "first",
            ),
            assigned_tenure_days=(
                "assigned_tenure_days",
                "first",
            ),
            TARGET=("TARGET", "first"),
            fallback_sampling=(
                "fallback_sampling",
                "first",
            ),
        )
    )

    customer_level = audit_values.merge(
        customer_level,
        on="ncodpers",
        how="inner",
        validate="one_to_one",
    )

    # Ensure final column order
    final_columns = (
        [
            "ncodpers",
            "fecha_alta",
            "last_snapshot_date",
            "assigned_cutoff",
            "assigned_tenure_days",
            "fallback_sampling",
        ]
        + MODEL_FEATURES
        + ["TARGET"]
    )

    customer_level = customer_level[
        final_columns
    ]

    return customer_level, meta_seed


# ==========================================================
# 14. Run five placebo seeds
# ==========================================================

summary_rows = []

for seed in SEEDS:
    print("\n" + "=" * 70)
    print(f"Đang tạo placebo dataset với seed = {seed}")
    print("=" * 70)

    placebo_data, cutoff_meta = (
        construct_placebo_dataset(
            longitudinal_df=df,
            meta_df=customer_meta,
            seed=seed,
        )
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        (
            "Santander_Placebo_"
            f"CustomerLevel_seed_{seed}.csv"
        ),
    )

    placebo_data.to_csv(
        output_file,
        index=False,
    )

    # ------------------------------------------------------
    # Audit placebo tenure distribution
    # ------------------------------------------------------

    adopter_assigned = cutoff_meta.loc[
        cutoff_meta["TARGET"].eq(1),
        "assigned_tenure_days",
    ].dropna()

    non_adopter_assigned = cutoff_meta.loc[
        cutoff_meta["TARGET"].eq(0),
        "assigned_tenure_days",
    ].dropna()

    ks_result = ks_2samp(
        adopter_assigned,
        non_adopter_assigned,
    )

    fallback_count = int(
        cutoff_meta.loc[
            cutoff_meta["TARGET"].eq(0),
            "fallback_sampling",
        ].sum()
    )

    positive_rate = placebo_data[
        "TARGET"
    ].mean()

    summary_rows.append({
        "seed": seed,
        "customers": len(placebo_data),
        "positive_customers": int(
            placebo_data["TARGET"].sum()
        ),
        "positive_rate": positive_rate,
        "adopter_tenure_mean": (
            adopter_assigned.mean()
        ),
        "adopter_tenure_median": (
            adopter_assigned.median()
        ),
        "nonadopter_tenure_mean": (
            non_adopter_assigned.mean()
        ),
        "nonadopter_tenure_median": (
            non_adopter_assigned.median()
        ),
        "tenure_KS": ks_result.statistic,
        "tenure_KS_pvalue": ks_result.pvalue,
        "fallback_count": fallback_count,
        "output_file": output_file,
    })

    print("Kích thước:", placebo_data.shape)
    print(
        "Số khách hàng duy nhất:",
        placebo_data["ncodpers"].nunique(),
    )
    print(
        "Phân bố TARGET:\n",
        placebo_data["TARGET"].value_counts(),
    )
    print(
        "Tỷ lệ TARGET:",
        round(positive_rate, 6),
    )
    print(
        "Adopter tenure mean:",
        round(adopter_assigned.mean(), 2),
    )
    print(
        "Non-adopter placebo tenure mean:",
        round(non_adopter_assigned.mean(), 2),
    )
    print(
        "KS giữa hai phân phối tenure:",
        round(ks_result.statistic, 4),
    )
    print(
        "Số trường hợp fallback:",
        fallback_count,
    )
    print("Đã lưu:", output_file)

    del placebo_data
    del cutoff_meta
    gc.collect()

# ==========================================================
# 15. Save audit summary
# ==========================================================

summary_df = pd.DataFrame(summary_rows)

summary_file = os.path.join(
    OUTPUT_DIR,
    "Santander_Placebo_Cutoff_Audit_Summary.csv",
)

summary_df.to_csv(
    summary_file,
    index=False,
)

print("\n" + "=" * 70)
print("HOÀN THÀNH 5 PLACEBO DATASETS")
print("=" * 70)

display(summary_df)

print("\nThư mục đầu ra:")
print(OUTPUT_DIR)

print("\nFile tổng hợp:")
print(summary_file)

Mounted at /content/drive
Số đặc trưng mô hình: 20

Đang đọc file:
/content/drive/MyDrive/Santander_CreditCard.csv

Kích thước dữ liệu gốc: (13647309, 25)
Sau khi giữ năm quan hệ đầu tiên: (1865323, 26)
Số khách hàng: 256342

Adopters không có dữ liệu trước event: 715

Phân phối adopter tenure:
count    1591.000000
mean      178.792583
std        97.665594
min        31.000000
25%        94.000000
50%       165.000000
75%       264.000000
max       364.000000
dtype: float64

Đang tạo placebo dataset với seed = 42
Kích thước: (255627, 27)
Số khách hàng duy nhất: 255627
Phân bố TARGET:
 TARGET
0    254036
1      1591
Name: count, dtype: int64
Tỷ lệ TARGET: 0.006224
Adopter tenure mean: 178.79
Non-adopter placebo tenure mean: 175.42
KS giữa hai phân phối tenure: 0.0462
Số trường hợp fallback: 8598
Đã lưu: /content/drive/MyDrive/Santander_Placebo_Cutoff_Datasets/Santander_Placebo_CustomerLevel_seed_42.csv

Đang tạo placebo dataset với seed = 52
Kích thước: (255627, 27)
Số khách hàng duy nh

,seed,customers,positive_customers,positive_rate,adopter_tenure_mean,adopter_tenure_median,nonadopter_tenure_mean,nonadopter_tenure_median,tenure_KS,tenure_KS_pvalue,fallback_count,output_file
0,42,255627,1591,0.006224,178.792583,165.0,175.419063,165.0,0.046168,0.002289,8598,/content/drive/MyDrive/Santander_Placebo_Cutof...
1,52,255627,1591,0.006224,178.792583,165.0,175.353544,165.0,0.046215,0.002258,8598,/content/drive/MyDrive/Santander_Placebo_Cutof...
2,62,255627,1591,0.006224,178.792583,165.0,175.147420,164.0,0.047154,0.001710,8598,/content/drive/MyDrive/Santander_Placebo_Cutof...
3,72,255627,1591,0.006224,178.792583,165.0,175.295923,165.0,0.046560,0.002040,8598,/content/drive/MyDrive/Santander_Placebo_Cutof...
4,82,255627,1591,0.006224,178.792583,165.0,175.146578,165.0,0.046384,0.002148,8598,/content/drive/MyDrive/Santander_Placebo_Cutof...



Thư mục đầu ra:
/content/drive/MyDrive/Santander_Placebo_Cutoff_Datasets

File tổng hợp:
/content/drive/MyDrive/Santander_Placebo_Cutoff_Datasets/Santander_Placebo_Cutoff_Audit_Summary.csv


HUẤN LUYỆN MÔ HÌNH PLACEBO

In [ ]:
# ==========================================================
# SANTANDER PLACEBO-CUTOFF
# XGBOOST + RANDOM OVERSAMPLING OVER FIVE PLACEBO DATASETS
#
# Only the placebo-cutoff assignment changes across datasets.
# Train/test split, preprocessing, ROS, and XGBoost are fixed.
# ==========================================================

# ==========================================================
# 1. Mount Google Drive
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

# ==========================================================
# 2. Install libraries
# ==========================================================

!pip install -q xgboost imbalanced-learn

# ==========================================================
# 3. Imports
# ==========================================================

import os
import gc
import warnings

import numpy as np
import pandas as pd

from scipy.stats import ks_2samp

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    brier_score_loss,
    confusion_matrix,
    classification_report
)

from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ==========================================================
# 4. Configuration
# ==========================================================

PLACEBO_DIR = (
    "/content/drive/MyDrive/"
    "Santander_Placebo_Cutoff_Datasets"
)

OUTPUT_DIR = os.path.join(
    PLACEBO_DIR,
    "XGBoost_ROS_FixedTraining_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# These seeds only identify the five placebo datasets
PLACEBO_SEEDS = [42, 52, 62, 72, 82]

TEST_SIZE = 0.20

# Fixed for all five runs
SPLIT_RANDOM_STATE = 42
ROS_RANDOM_STATE = 42
MODEL_RANDOM_STATE = 42

TARGET_COL = "TARGET"
ID_COL = "ncodpers"

# ==========================================================
# 5. Exact 20 predictor variables
# ==========================================================

MODEL_FEATURES = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "age",
    "ind_nuevo",
    "antiguedad",
    "indrel",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "conyuemp",
    "indfall",
    "tipodom",
    "cod_prov",
    "nomprov",
    "ind_actividad_cliente",
    "renta",
    "segmento",
    "canal_entrada"
]

assert len(MODEL_FEATURES) == 20

# Numeric predictors
NUMERIC_FEATURES = [
    "age",
    "antiguedad",
    "renta"
]

# Remaining predictors are treated as categorical
CATEGORICAL_FEATURES = [
    col
    for col in MODEL_FEATURES
    if col not in NUMERIC_FEATURES
]

assert len(NUMERIC_FEATURES) == 3
assert len(CATEGORICAL_FEATURES) == 17

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total predictors:", len(MODEL_FEATURES))
print("Target:", TARGET_COL)

# ==========================================================
# 6. Find placebo dataset
# ==========================================================

def find_placebo_file(placebo_seed):
    filename = (
        "Santander_Placebo_"
        f"CustomerLevel_seed_{placebo_seed}.csv"
    )

    filepath = os.path.join(
        PLACEBO_DIR,
        filename
    )

    if os.path.exists(filepath):
        return filepath

    for root, _, files in os.walk(PLACEBO_DIR):
        if filename in files:
            return os.path.join(root, filename)

    raise FileNotFoundError(
        f"Không tìm thấy file placebo seed {placebo_seed}: "
        f"{filename}"
    )

# ==========================================================
# 7. Validate input dataset
# ==========================================================

def validate_dataset(data, placebo_seed):
    required_columns = (
        [ID_COL]
        + MODEL_FEATURES
        + [TARGET_COL]
    )

    missing_columns = [
        col
        for col in required_columns
        if col not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Seed {placebo_seed} thiếu các cột: "
            f"{missing_columns}"
        )

    if data[ID_COL].duplicated().any():
        duplicate_count = int(
            data[ID_COL].duplicated().sum()
        )

        raise ValueError(
            f"Seed {placebo_seed} có "
            f"{duplicate_count} customer ID bị trùng."
        )

    target_numeric = pd.to_numeric(
        data[TARGET_COL],
        errors="coerce"
    )

    if target_numeric.isna().any():
        raise ValueError(
            f"Seed {placebo_seed}: TARGET có giá trị "
            "không chuyển được sang số."
        )

    target_values = set(
        target_numeric.astype(int).unique()
    )

    if not target_values.issubset({0, 1}):
        raise ValueError(
            f"Seed {placebo_seed}: TARGET không phải nhị phân."
        )

    audit_columns = [
        "fecha_alta",
        "last_snapshot_date",
        "assigned_cutoff",
        "assigned_tenure_days",
        "fallback_sampling"
    ]

    leaked_audit_features = [
        col
        for col in audit_columns
        if col in MODEL_FEATURES
    ]

    if leaked_audit_features:
        raise ValueError(
            "Các cột audit bị đưa vào mô hình: "
            f"{leaked_audit_features}"
        )

# ==========================================================
# 8. Normalize feature types
# ==========================================================

def normalize_feature_types(X):
    """
    Numeric features:
        Convert to numeric; invalid values become NaN.

    Categorical features:
        Standardize missing values and convert all
        non-missing values to string.
    """

    X = X.copy()

    for col in NUMERIC_FEATURES:
        X[col] = pd.to_numeric(
            X[col],
            errors="coerce"
        )

    missing_tokens = {
        "": np.nan,
        " ": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
        "NAN": np.nan,
        "None": np.nan,
        "NONE": np.nan,
        "null": np.nan,
        "NULL": np.nan
    }

    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].replace(
            missing_tokens
        )

        X[col] = X[col].apply(
            lambda value: (
                str(value).strip()
                if pd.notna(value)
                else np.nan
            )
        )

    return X

# ==========================================================
# 9. Preprocessing
# ==========================================================

def preprocess_train_test(
    X_train,
    X_test
):
    """
    Fit all preprocessing components only on training data.
    """

    # ------------------------------------------------------
    # Numeric imputation
    # ------------------------------------------------------

    numeric_imputer = SimpleImputer(
        strategy="median"
    )

    X_train_numeric = pd.DataFrame(
        numeric_imputer.fit_transform(
            X_train[NUMERIC_FEATURES]
        ),
        columns=NUMERIC_FEATURES,
        index=X_train.index
    )

    X_test_numeric = pd.DataFrame(
        numeric_imputer.transform(
            X_test[NUMERIC_FEATURES]
        ),
        columns=NUMERIC_FEATURES,
        index=X_test.index
    )

    # ------------------------------------------------------
    # Categorical imputation
    # ------------------------------------------------------

    categorical_imputer = SimpleImputer(
        strategy="most_frequent"
    )

    X_train_categorical_raw = pd.DataFrame(
        categorical_imputer.fit_transform(
            X_train[CATEGORICAL_FEATURES]
        ),
        columns=CATEGORICAL_FEATURES,
        index=X_train.index
    )

    X_test_categorical_raw = pd.DataFrame(
        categorical_imputer.transform(
            X_test[CATEGORICAL_FEATURES]
        ),
        columns=CATEGORICAL_FEATURES,
        index=X_test.index
    )

    # ------------------------------------------------------
    # Ordinal encoding
    # ------------------------------------------------------

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    X_train_categorical = pd.DataFrame(
        encoder.fit_transform(
            X_train_categorical_raw
        ),
        columns=CATEGORICAL_FEATURES,
        index=X_train.index
    )

    X_test_categorical = pd.DataFrame(
        encoder.transform(
            X_test_categorical_raw
        ),
        columns=CATEGORICAL_FEATURES,
        index=X_test.index
    )

    # ------------------------------------------------------
    # Combine numeric and categorical features
    # ------------------------------------------------------

    X_train_processed = pd.concat(
        [
            X_train_numeric,
            X_train_categorical
        ],
        axis=1
    )

    X_test_processed = pd.concat(
        [
            X_test_numeric,
            X_test_categorical
        ],
        axis=1
    )

    # Restore exact feature order
    X_train_processed = X_train_processed[
        MODEL_FEATURES
    ]

    X_test_processed = X_test_processed[
        MODEL_FEATURES
    ]

    assert X_train_processed.shape[1] == 20
    assert X_test_processed.shape[1] == 20

    assert list(
        X_train_processed.columns
    ) == MODEL_FEATURES

    assert list(
        X_test_processed.columns
    ) == MODEL_FEATURES

    return (
        X_train_processed,
        X_test_processed
    )

# ==========================================================
# 10. KS statistic
# ==========================================================

def calculate_ks(
    y_true,
    probabilities
):
    y_true = np.asarray(
        y_true,
        dtype=int
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )

    positive_scores = probabilities[
        y_true == 1
    ]

    negative_scores = probabilities[
        y_true == 0
    ]

    if (
        len(positive_scores) == 0
        or len(negative_scores) == 0
    ):
        return np.nan

    return float(
        ks_2samp(
            positive_scores,
            negative_scores
        ).statistic
    )

# ==========================================================
# 11. Specificity
# ==========================================================

def calculate_specificity(
    y_true,
    y_pred
):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    if (tn + fp) == 0:
        return np.nan

    return tn / (tn + fp)

# ==========================================================
# 12. Ranking metrics
# ==========================================================

def calculate_ranking_metrics(
    y_true,
    probabilities,
    fractions=(0.10, 0.20, 0.30)
):
    ranking_data = pd.DataFrame({
        "TARGET": np.asarray(
            y_true,
            dtype=int
        ),
        "probability": np.asarray(
            probabilities,
            dtype=float
        )
    })

    ranking_data = ranking_data.sort_values(
        "probability",
        ascending=False
    ).reset_index(drop=True)

    total_customers = len(ranking_data)

    total_positives = int(
        ranking_data["TARGET"].sum()
    )

    base_rate = float(
        ranking_data["TARGET"].mean()
    )

    metrics = {}

    for fraction in fractions:
        selected_count = int(
            np.ceil(
                total_customers * fraction
            )
        )

        top_group = ranking_data.iloc[
            :selected_count
        ]

        selected_positives = int(
            top_group["TARGET"].sum()
        )

        precision_at_k = (
            selected_positives / selected_count
            if selected_count > 0
            else np.nan
        )

        capture_at_k = (
            selected_positives / total_positives
            if total_positives > 0
            else np.nan
        )

        lift_at_k = (
            precision_at_k / base_rate
            if base_rate > 0
            else np.nan
        )

        percentage = int(
            round(fraction * 100)
        )

        metrics[
            f"Precision@{percentage}%"
        ] = precision_at_k

        metrics[
            f"Capture@{percentage}%"
        ] = capture_at_k

        metrics[
            f"Lift@{percentage}%"
        ] = lift_at_k

    return metrics

# ==========================================================
# 13. Create one fixed train/test partition
# ==========================================================

reference_file = find_placebo_file(
    PLACEBO_SEEDS[0]
)

reference_data = pd.read_csv(
    reference_file,
    usecols=[
        ID_COL,
        TARGET_COL
    ],
    low_memory=False
)

reference_data[TARGET_COL] = pd.to_numeric(
    reference_data[TARGET_COL],
    errors="raise"
).astype(int)

train_ids_array, test_ids_array = train_test_split(
    reference_data[ID_COL].to_numpy(),
    test_size=TEST_SIZE,
    random_state=SPLIT_RANDOM_STATE,
    stratify=reference_data[TARGET_COL].to_numpy()
)

train_ids = set(
    train_ids_array.tolist()
)

test_ids = set(
    test_ids_array.tolist()
)

reference_test = reference_data[
    reference_data[ID_COL].isin(test_ids)
]

print("\nFixed train/test partition")
print("Training customers:", len(train_ids))
print("Test customers:", len(test_ids))
print(
    "Test positive customers:",
    int(reference_test[TARGET_COL].sum())
)
print(
    "Test positive rate:",
    round(
        reference_test[TARGET_COL].mean(),
        6
    )
)

del reference_data
del reference_test
gc.collect()

# ==========================================================
# 14. Train on five placebo datasets
# ==========================================================

result_rows = []

for placebo_seed in PLACEBO_SEEDS:

    print("\n" + "=" * 78)
    print(
        f"PLACEBO DATASET SEED {placebo_seed}"
    )
    print(
        "Fixed split seed = 42 | "
        "Fixed ROS seed = 42 | "
        "Fixed XGBoost seed = 42"
    )
    print("=" * 78)

    input_file = find_placebo_file(
        placebo_seed
    )

    print("Reading:")
    print(input_file)

    data = pd.read_csv(
        input_file,
        low_memory=False
    )

    validate_dataset(
        data,
        placebo_seed
    )

    data[TARGET_COL] = pd.to_numeric(
        data[TARGET_COL],
        errors="raise"
    ).astype(int)

    # ------------------------------------------------------
    # Fixed train/test customers
    # ------------------------------------------------------

    train_data = data[
        data[ID_COL].isin(train_ids)
    ].copy()

    test_data = data[
        data[ID_COL].isin(test_ids)
    ].copy()

    if len(train_data) != len(train_ids):
        raise ValueError(
            f"Seed {placebo_seed}: "
            "training customer count mismatch."
        )

    if len(test_data) != len(test_ids):
        raise ValueError(
            f"Seed {placebo_seed}: "
            "test customer count mismatch."
        )

    # ------------------------------------------------------
    # Exact 20 predictors and one target
    # ------------------------------------------------------

    X_train = train_data[
        MODEL_FEATURES
    ].copy()

    X_test = test_data[
        MODEL_FEATURES
    ].copy()

    y_train = train_data[
        TARGET_COL
    ].astype(int)

    y_test = test_data[
        TARGET_COL
    ].astype(int)

    assert X_train.shape[1] == 20
    assert X_test.shape[1] == 20

    # ------------------------------------------------------
    # Normalize data types
    # ------------------------------------------------------

    X_train = normalize_feature_types(
        X_train
    )

    X_test = normalize_feature_types(
        X_test
    )

    print("Raw train shape:", X_train.shape)
    print("Raw test shape:", X_test.shape)
    print(
        "Training positive rate before ROS:",
        round(
            y_train.mean(),
            6
        )
    )
    print(
        "Test positive rate:",
        round(
            y_test.mean(),
            6
        )
    )

    # ------------------------------------------------------
    # Preprocessing
    # ------------------------------------------------------

    (
        X_train_processed,
        X_test_processed
    ) = preprocess_train_test(
        X_train,
        X_test
    )

    print(
        "Processed train shape:",
        X_train_processed.shape
    )
    print(
        "Processed test shape:",
        X_test_processed.shape
    )

    # ------------------------------------------------------
    # Random Oversampling: fixed seed 42
    # ------------------------------------------------------

    ros = RandomOverSampler(
        random_state=ROS_RANDOM_STATE
    )

    X_train_ros, y_train_ros = (
        ros.fit_resample(
            X_train_processed,
            y_train
        )
    )

    print(
        "Class distribution after ROS:"
    )
    print(
        pd.Series(
            y_train_ros
        ).value_counts()
    )

    # ------------------------------------------------------
    # XGBoost: fixed configuration
    # ------------------------------------------------------

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=MODEL_RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        verbosity=0
    )

    model.fit(
        X_train_ros,
        y_train_ros
    )

    # ------------------------------------------------------
    # Prediction
    # ------------------------------------------------------

    probabilities = model.predict_proba(
        X_test_processed
    )[:, 1]

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    # ------------------------------------------------------
    # Classification metrics
    # ------------------------------------------------------

    auc_value = roc_auc_score(
        y_test,
        probabilities
    )

    accuracy_value = accuracy_score(
        y_test,
        predictions
    )

    recall_value = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    precision_value = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1_value = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    specificity_value = (
        calculate_specificity(
            y_test,
            predictions
        )
    )

    ks_value = calculate_ks(
        y_test,
        probabilities
    )

    brier_value = brier_score_loss(
        y_test,
        probabilities
    )

    ranking_metrics = (
        calculate_ranking_metrics(
            y_true=y_test,
            probabilities=probabilities
        )
    )

    # ------------------------------------------------------
    # Save predictions
    # ------------------------------------------------------

    prediction_output = pd.DataFrame({
        ID_COL: test_data[
            ID_COL
        ].to_numpy(),
        "TARGET": y_test.to_numpy(),
        "predicted_probability": probabilities,
        "predicted_class_0_5": predictions
    })

    prediction_file = os.path.join(
        OUTPUT_DIR,
        (
            "Santander_Placebo_"
            f"Predictions_seed_{placebo_seed}.csv"
        )
    )

    prediction_output.to_csv(
        prediction_file,
        index=False
    )

    # ------------------------------------------------------
    # Save metrics
    # ------------------------------------------------------

    result_row = {
        "placebo_seed": placebo_seed,
        "split_random_state": SPLIT_RANDOM_STATE,
        "ros_random_state": ROS_RANDOM_STATE,
        "model_random_state": MODEL_RANDOM_STATE,
        "train_customers": len(train_data),
        "test_customers": len(test_data),
        "test_positive_customers": int(
            y_test.sum()
        ),
        "test_positive_rate": float(
            y_test.mean()
        ),
        "AUC": auc_value,
        "Accuracy": accuracy_value,
        "Recall": recall_value,
        "Precision_threshold_0.5": (
            precision_value
        ),
        "F1": f1_value,
        "Specificity": specificity_value,
        "KS": ks_value,
        "Brier": brier_value,
        **ranking_metrics,
        "prediction_file": prediction_file
    }

    result_rows.append(
        result_row
    )

    print("\nResults")
    print(f"AUC             : {auc_value:.6f}")
    print(f"Accuracy        : {accuracy_value:.6f}")
    print(f"Recall          : {recall_value:.6f}")
    print(f"Precision@0.5   : {precision_value:.6f}")
    print(f"F1              : {f1_value:.6f}")
    print(f"Specificity     : {specificity_value:.6f}")
    print(f"KS              : {ks_value:.6f}")
    print(f"Brier           : {brier_value:.6f}")

    print(
        "Precision@10%   : "
        f"{ranking_metrics['Precision@10%']:.6f}"
    )
    print(
        "Lift@10%        : "
        f"{ranking_metrics['Lift@10%']:.6f}"
    )
    print(
        "Capture@30%     : "
        f"{ranking_metrics['Capture@30%']:.6f}"
    )

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            predictions,
            digits=4,
            zero_division=0
        )
    )

    # ------------------------------------------------------
    # Memory cleanup
    # ------------------------------------------------------

    del data
    del train_data
    del test_data
    del X_train
    del X_test
    del X_train_processed
    del X_test_processed
    del X_train_ros
    del y_train_ros
    del model
    del prediction_output

    gc.collect()

# ==========================================================
# 15. Save detailed results
# ==========================================================

results_df = pd.DataFrame(
    result_rows
)

detailed_results_file = os.path.join(
    OUTPUT_DIR,
    "Santander_Placebo_"
    "XGBoost_ROS_FixedTraining_Detailed.csv"
)

results_df.to_csv(
    detailed_results_file,
    index=False
)

print("\n" + "=" * 78)
print("DETAILED RESULTS OVER FIVE PLACEBO DATASETS")
print("=" * 78)

display(results_df)

# ==========================================================
# 16. Mean ± sample SD
# ==========================================================

REPORT_METRICS = [
    "AUC",
    "Accuracy",
    "Recall",
    "Precision_threshold_0.5",
    "F1",
    "Specificity",
    "KS",
    "Brier",
    "Precision@10%",
    "Lift@10%",
    "Capture@10%",
    "Precision@20%",
    "Lift@20%",
    "Capture@20%",
    "Precision@30%",
    "Lift@30%",
    "Capture@30%"
]

summary_rows = []

for metric in REPORT_METRICS:

    mean_value = results_df[
        metric
    ].mean()

    sd_value = results_df[
        metric
    ].std(ddof=1)

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "SD": sd_value,
        "Mean ± SD": (
            f"{mean_value:.4f} ± "
            f"{sd_value:.4f}"
        )
    })

summary_df = pd.DataFrame(
    summary_rows
)

summary_file = os.path.join(
    OUTPUT_DIR,
    "Santander_Placebo_"
    "XGBoost_ROS_FixedTraining_Mean_SD.csv"
)

summary_df.to_csv(
    summary_file,
    index=False
)

print("\n" + "=" * 78)
print("MEAN ± SD OVER FIVE PLACEBO DATASETS")
print("=" * 78)

display(summary_df)

# ==========================================================
# 17. Paper-ready ranking table
# ==========================================================

PAPER_METRICS = [
    "AUC",
    "KS",
    "Precision@10%",
    "Lift@10%",
    "Capture@30%"
]

paper_table = (
    summary_df[
        summary_df["Metric"].isin(
            PAPER_METRICS
        )
    ]
    .set_index("Metric")
    .loc[PAPER_METRICS]
    .reset_index()
)

paper_table_file = os.path.join(
    OUTPUT_DIR,
    "Santander_Placebo_"
    "Paper_Ready_Ranking_Table.csv"
)

paper_table.to_csv(
    paper_table_file,
    index=False
)

print("\n" + "=" * 78)
print("PAPER-READY TABLE")
print("=" * 78)

display(
    paper_table[
        [
            "Metric",
            "Mean ± SD"
        ]
    ]
)

# ==========================================================
# 18. Print manuscript-ready sentence
# ==========================================================

summary_lookup = (
    summary_df
    .set_index("Metric")[
        "Mean ± SD"
    ]
    .to_dict()
)

print("\nManuscript-ready result:\n")

print(
    "Across five independently reconstructed "
    "placebo-cutoff datasets, XGBoost + ROS achieved "
    "a mean test AUC of "
    f"{summary_lookup['AUC']}, "
    "a KS statistic of "
    f"{summary_lookup['KS']}, "
    "a Precision@10% of "
    f"{summary_lookup['Precision@10%']}, "
    "a Lift@10% of "
    f"{summary_lookup['Lift@10%']}, "
    "and a Capture@30% of "
    f"{summary_lookup['Capture@30%']}."
)

print("\nSaved files:")
print(detailed_results_file)
print(summary_file)
print(paper_table_file)

print("\nOutput directory:")
print(OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Numeric features: 3
Categorical features: 17
Total predictors: 20
Target: TARGET

Fixed train/test partition
Training customers: 204501
Test customers: 51126
Test positive customers: 318
Test positive rate: 0.00622

PLACEBO DATASET SEED 42
Fixed split seed = 42 | Fixed ROS seed = 42 | Fixed XGBoost seed = 42
Reading:
/content/drive/MyDrive/Santander_Placebo_Cutoff_Datasets/Santander_Placebo_CustomerLevel_seed_42.csv
Raw train shape: (204501, 20)
Raw test shape: (51126, 20)
Training positive rate before ROS: 0.006225
Test positive rate: 0.00622
Processed train shape: (204501, 20)
Processed test shape: (51126, 20)
Class distribution after ROS:
TARGET
0    203228
1    203228
Name: count, dtype: int64

Results
AUC             : 0.892821
Accuracy        : 0.828580
Recall          : 0.767296
Precision@0.5   : 0.027311
F1              : 0.052745
Specificity     : 0.

,placebo_seed,split_random_state,ros_random_state,model_random_state,train_customers,test_customers,test_positive_customers,test_positive_rate,AUC,Accuracy,...,Precision@10%,Capture@10%,Lift@10%,Precision@20%,Capture@20%,Lift@20%,Precision@30%,Capture@30%,Lift@30%,prediction_file
0,42,42,42,42,204501,51126,318,0.00622,0.892821,0.828580,...,0.034226,0.550314,5.502714,0.025425,0.817610,4.087730,0.019755,0.952830,3.176059,/content/drive/MyDrive/Santander_Placebo_Cutof...
1,52,42,42,42,204501,51126,318,0.00622,0.891340,0.827954,...,0.034226,0.550314,5.502714,0.025230,0.811321,4.056286,0.019820,0.955975,3.186541,/content/drive/MyDrive/Santander_Placebo_Cutof...
2,62,42,42,42,204501,51126,318,0.00622,0.891225,0.828913,...,0.032271,0.518868,5.188273,0.025523,0.820755,4.103453,0.019820,0.955975,3.186541,/content/drive/MyDrive/Santander_Placebo_Cutof...
3,72,42,42,42,204501,51126,318,0.00622,0.892223,0.828267,...,0.034226,0.550314,5.502714,0.025523,0.820755,4.103453,0.019820,0.955975,3.186541,/content/drive/MyDrive/Santander_Placebo_Cutof...
4,82,42,42,42,204501,51126,318,0.00622,0.891426,0.828346,...,0.033053,0.531447,5.314050,0.025132,0.808176,4.040564,0.019755,0.952830,3.176059,/content/drive/MyDrive/Santander_Placebo_Cutof...



MEAN ± SD OVER FIVE PLACEBO DATASETS


,Metric,Mean,SD,Mean ± SD
0,AUC,0.891807,0.000690,0.8918 ± 0.0007
1,Accuracy,0.828412,0.000358,0.8284 ± 0.0004
2,Recall,0.764151,0.005883,0.7642 ± 0.0059
3,Precision_threshold_0.5,0.027180,0.000232,0.0272 ± 0.0002
4,F1,0.052492,0.000446,0.0525 ± 0.0004
5,Specificity,0.828814,0.000343,0.8288 ± 0.0003
6,KS,0.676738,0.004216,0.6767 ± 0.0042
7,Brier,0.102435,0.000404,0.1024 ± 0.0004
8,Precision@10%,0.033601,0.000901,0.0336 ± 0.0009
9,Lift@10%,5.402093,0.144779,5.4021 ± 0.1448



PAPER-READY TABLE


,Metric,Mean ± SD
0,AUC,0.8918 ± 0.0007
1,KS,0.6767 ± 0.0042
2,Precision@10%,0.0336 ± 0.0009
3,Lift@10%,5.4021 ± 0.1448
4,Capture@30%,0.9547 ± 0.0017



Manuscript-ready result:

Across five independently reconstructed placebo-cutoff datasets, XGBoost + ROS achieved a mean test AUC of 0.8918 ± 0.0007, a KS statistic of 0.6767 ± 0.0042, a Precision@10% of 0.0336 ± 0.0009, a Lift@10% of 5.4021 ± 0.1448, and a Capture@30% of 0.9547 ± 0.0017.

Saved files:
/content/drive/MyDrive/Santander_Placebo_Cutoff_Datasets/XGBoost_ROS_FixedTraining_Results/Santander_Placebo_XGBoost_ROS_FixedTraining_Detailed.csv
/content/drive/MyDrive/Santander_Placebo_Cutoff_Datasets/XGBoost_ROS_FixedTraining_Results/Santander_Placebo_XGBoost_ROS_FixedTraining_Mean_SD.csv
/content/drive/MyDrive/Santander_Placebo_Cutoff_Datasets/XGBoost_ROS_FixedTraining_Results/Santander_Placebo_Paper_Ready_Ranking_Table.csv

Output directory:
/content/drive/MyDrive/Santander_Placebo_Cutoff_Datasets/XGBoost_ROS_FixedTraining_Results
